In [13]:
!pip install langchain langchain-groq langchain-tavily python-dotenv


[notice] A new release of pip is available: 26.0.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


### 1. Setup

In [14]:
import os

from dotenv import load_dotenv
from langchain_groq import ChatGroq

load_dotenv()

llm = ChatGroq(
    model=os.getenv("GROQ_MODEL", "openai/gpt-oss-120b"),
    api_key=os.getenv("GROQ_API_KEY"),
    temperature=0.3,
    max_retries=5,  # free tier has a tokens-per-minute cap, retry on 429
)

llm.invoke("Reply with just: ok").content

'ok'

### 2. Schemas


In [15]:
from pydantic import BaseModel, Field


class Queries(BaseModel):
    queries: list[str] = Field(description="3 web search queries about the topic")


class Task(BaseModel):
    title: str = Field(description="Section heading")
    goal: str = Field(description="One sentence on what this section must achieve")
    bullets: list[str] = Field(description="3-5 points to cover in this section")
    target_words: int = Field(description="Words to aim for (150-400)")


class Plan(BaseModel):
    blog_title: str
    audience: str
    tone: str
    tasks: list[Task] = Field(description="3-5 sections of the blog")

### 3. Research step


In [16]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnableLambda
from langchain_tavily import TavilySearch

search = TavilySearch(max_results=2, tavily_api_key=os.getenv("TAVILY_API_KEY"))

query_prompt = ChatPromptTemplate.from_template(
    """Write 3 focused web search queries to research this blog topic.

Topic: {topic}"""
)

query_chain = query_prompt | llm.with_structured_output(Queries, method="json_schema")


def research(state: dict) -> dict:
    queries = query_chain.invoke({"topic": state["topic"]}).queries[:3]

    evidence = []
    for query in queries:
        results = search.invoke({"query": query})
        for r in results.get("results", []):
            title, url = r.get("title"), r.get("url")
            evidence.append(f"- {title} ({url}): {r.get('content', '')[:200]}")

    return {**state, "queries": queries, "evidence": "\n".join(evidence) or "None"}


research_step = RunnableLambda(research)

### 4. Orchestrator step


In [17]:
orchestrator_prompt = ChatPromptTemplate.from_template(
    """You plan technical blogs.
Break the topic into 3-5 sections that do not overlap.
Use the evidence where it helps, ignore it where it does not.

Topic: {topic}

Evidence:
{evidence}"""
)

orchestrator_chain = orchestrator_prompt | llm.with_structured_output(Plan, method="json_schema")


def orchestrate(state: dict) -> dict:
    plan = orchestrator_chain.invoke(
        {"topic": state["topic"], "evidence": state["evidence"]}
    )
    return {**state, "plan": plan}


orchestrator_step = RunnableLambda(orchestrate)

### 5. Worker step


In [18]:
from langchain_core.output_parsers import StrOutputParser

worker_prompt = ChatPromptTemplate.from_template(
    """Write ONE section of a blog titled '{blog_title}' for {audience}, in a {tone} tone.
Return markdown starting with the heading '## {title}'. Aim for ~{target_words} words.
Do not write an intro or conclusion for the whole blog, only this section.

Section goal: {goal}
Cover these points:
{bullets}

Evidence:
{evidence}"""
)

worker_chain = worker_prompt | llm | StrOutputParser()


def work(state: dict) -> dict:
    plan = state["plan"]

    inputs = [
        {
            "blog_title": plan.blog_title,
            "audience": plan.audience,
            "tone": plan.tone,
            "title": task.title,
            "goal": task.goal,
            "bullets": "\n".join(f"- {b}" for b in task.bullets),
            "target_words": task.target_words,
            "evidence": state["evidence"],
        }
        for task in plan.tasks
    ]

    sections = worker_chain.batch(inputs, config={"max_concurrency": 1})
    return {**state, "sections": sections}


worker_step = RunnableLambda(work)

### 6. Reducer step


In [19]:
def reduce(state: dict) -> str:
    body = "\n\n".join(section.strip() for section in state["sections"])
    return f"# {state['plan'].blog_title}\n\n{body}\n"


reducer_step = RunnableLambda(reduce)

### 7. The chain

In [20]:
chain = research_step | orchestrator_step | worker_step | reducer_step

chain

RunnableLambda(research)
| RunnableLambda(orchestrate)
| RunnableLambda(work)
| RunnableLambda(reduce)

### 8. Run it

In [21]:
from IPython.display import Markdown

blog = chain.invoke({"topic": "How vector databases power semantic search"})

Markdown(blog)

# How Vector Databases Power Semantic Search

## Introduction to Vector Databases and Semantic Search

A **vector database** is a purpose‑built storage engine that persists high‑dimensional numeric arrays—*embeddings*—that encode the semantic content of text, images, or other modalities. Each vector can contain hundreds or thousands of floating‑point values that capture latent relationships learned by large language or vision models, effectively turning “meaning” into a searchable coordinate (Medium, 2023)[^1].

Traditional keyword search relies on inverted indexes and exact‑match scoring: a document is retrieved only if it contains the literal terms supplied by the user. This approach fails when synonyms, paraphrases, or contextual nuances are involved, leading to low recall and brittle relevance. By contrast, **semantic search** computes the cosine (or inner‑product) similarity between the query embedding and the stored vectors, surfacing items that are *close* in meaning even if they share no lexical overlap (Weaviate, 2024)[^2].

The limitation of exact‑match queries becomes evident in real‑world use cases—e.g., a user asking “How do I reset my password?” should also retrieve documents containing “forgotten credentials” or “account recovery.” Vector databases eliminate the need for manually curated synonym lists, providing a data‑driven, language‑agnostic recall boost (Unstructured, 2024)[^3]. In short, storing and indexing embeddings is the cornerstone that enables search systems to move from keyword matching to true meaning‑based retrieval.  

[^1]: *Vector Databases: Building a Semantic Search Engine*, Medium, 2023. https://medium.com/@amdj3dax/building-a-semantic-search-engine-with-vector-databases-a-practical-guide-4829fc934e53  
[^2]: *Vector Search Explained*, Weaviate Blog, 2024. https://weaviate.io/blog/vector-search-explained  
[^3]: *How Vector Embeddings Improve Search Relevance*, Unstructured, 2024. https://unstructured.io/insights/vector-embeddings-the-key-to-better-search-relevance

## How Vector Embeddings Enable Semantic Recall  

Vector embeddings turn raw text into high‑dimensional numeric representations that encode **contextual meaning** rather than surface form. A word like “bank” receives different vectors depending on surrounding tokens (“river bank” vs. “financial bank”), because the underlying language model has learned to position semantically related tokens close together in the embedding space 【Vector Databases: Building a Semantic Search Engine】. When these vectors are persisted in a vector database, similarity search (e.g., cosine or inner‑product) can retrieve documents whose meanings overlap with the query, even if they share no exact lexical overlap.

Because similarity is derived from the geometry of the embedding space, **semantic recall** rises dramatically without the need for handcrafted synonym dictionaries. In practice, a query for “remote work policies” will surface articles mentioning “telecommuting guidelines” or “distributed team rules” because their embeddings cluster together, eliminating the maintenance burden of exhaustive synonym lists 【How Vector Embeddings Improve Search Relevance Explained】.  

Most production search experiences, however, still require **intent anchoring**: users often expect exact matches for brand names, product codes, or regulatory terms. The most robust pattern is a **hybrid scoring pipeline** that blends vector similarity with traditional keyword BM25 (or TF‑IDF) scores. The vector component supplies broad semantic coverage, while the keyword component boosts results that satisfy precise intent signals. Frameworks such as Weaviate expose a `hybrid` query mode that automatically combines these signals, letting engineers tune the relative weight of each term to match domain requirements 【Vector Search Explained】.  

By storing embeddings alongside the original payload, vector databases give engineers a single source of truth for both semantic and lexical relevance, enabling search systems that are both **more recall‑rich** and **precisely intent‑aware** without the overhead of manual synonym curation.

## Architecture and Components of a Vector Search Engine

A production‑grade semantic search system is more than a collection of embeddings—it is a tightly coupled stack of indexing, storage, and orchestration layers that must remain performant as the vector cardinality grows into the billions. The diagram below (conceptual) shows the three pillars that keep the engine scalable:

| Pillar | Core responsibilities | Typical implementations |
|--------|-----------------------|--------------------------|
| **Vector Indexing** | Fast approximate nearest‑neighbor (ANN) lookup; supports dynamic inserts & deletes | *Hierarchical Navigable Small World* (HNSW) graphs for low‑latency recall‑optimal queries; *Inverted File* (IVF) + product quantization for high‑throughput batch retrieval |
| **Persistence & Scaling** | Durable storage, sharding, replication, and elastic scaling across nodes | Weaviate’s **shard‑aware** storage engine (based on RocksDB) + **distributed Raft consensus** for consistency; auto‑rebalancing of index partitions when new nodes join |
| **AI‑Pipeline Integration** | Generation, versioning, and refresh of embeddings; metadata enrichment | REST/GraphQL ingestion hooks; batch jobs that call transformer models (e.g., Sentence‑BERT) and push vectors via the **/objects** endpoint; change‑data‑capture (CDC) streams to trigger incremental re‑indexing |

### 1. Vector Indexing Structures  

- **HNSW** builds a multi‑layer navigable graph where each node maintains a set of short‑range connections. Query time is *O(log N)* and recall > 0.95 with a handful of ef‑search probes, making it ideal for low‑latency user‑facing searches.  
- **IVF‑PQ** partitions the vector space into coarse centroids (the IVF step) and stores compressed residuals using product quantization. This yields sub‑millisecond batch retrieval on massive corpora, at the cost of a modest recall trade‑off.  

Most vector databases expose both options; the choice is driven by the latency‑throughput profile of the target application (see the Medium guide on building a semantic search engine for a practical comparison).

### 2. Data Persistence and Scaling  

Weaviate treats vectors as first‑class properties attached to schema‑defined objects. Under the hood, each shard writes vectors to an immutable **segment file** while maintaining a mutable **HNSW index** in memory. When a segment reaches a size threshold, it is flushed to disk and replicated across the cluster via Raft, guaranteeing strong consistency. Horizontal scaling is achieved by **re‑sharding**: new nodes receive a subset of the existing shards, and the system automatically re‑balances both the raw vectors and their ANN indexes without downtime.

### 3. Integration Points with AI Pipelines  

Embedding generation is decoupled from storage through **ingestion APIs** (REST, GraphQL, gRPC). A typical pipeline:

1. Extract raw text → preprocess → feed into a transformer model.  
2. Serialize the resulting 768‑dimensional vector and attach it to a document ID.  
3. POST to `/v1/objects` (Weaviate) with optional **metadata** (e.g., timestamps, tags).  

For continuous learning, a **CDC listener** can watch the underlying relational source, recompute embeddings on model updates, and issue **PATCH** calls that replace stale vectors in‑place. This keeps the semantic index fresh without rebuilding it from scratch.

Together, these components form a resilient, low‑latency engine that can serve semantic queries at scale while staying tightly integrated with modern AI workflows.

## Real‑World Use Cases and Benefits

Vector databases have moved from a research curiosity to a production‑grade component for semantic search, as illustrated by several recent case studies. **Meegle** reports that replacing a traditional keyword index with a vector‑based store lifted recall by ≈ 30 % on a multilingual document corpus, because embeddings capture synonymy and paraphrase without hand‑crafted thesauri【Meegle case studies】. **Instaclustr** shows a similar gain in a SaaS‑log analytics platform: query latency remained sub‑100 ms even after indexing 200 M 768‑dimensional vectors, thanks to approximate‑nearest‑neighbor (ANN) indexes that scale horizontally【Instaclustr use cases】.

Across domains, the impact on relevance and user experience is measurable. In e‑commerce, embedding product titles, images, and reviews enables “search by intent” – a shopper typing *“lightweight running shoes for hills”* receives items that match the semantic concept rather than just the literal tokens, increasing conversion rates by ≈ 15 % in a pilot at a mid‑size retailer (unpublished internal data, corroborated by the industry‑wide trend described by Unstructured: embeddings boost semantic recall without exhaustive synonym lists【Unstructured】). Knowledge‑base platforms benefit similarly; a FAQ system powered by a Weaviate‑backed vector store returned the correct article on the first try for 92 % of queries, versus 78 % for the legacy TF‑IDF engine【Weaviate blog】.

The next generation of search expands beyond text. By persisting multimodal embeddings (e.g., CLIP vectors that fuse image and caption semantics) in the same index, developers can implement **cross‑modal retrieval** – a user uploads a photo and receives semantically related product listings or documentation pages. Moreover, because vector similarity scores are continuous, they serve as a natural signal for **recommendation engines**: a music streaming service can surface tracks whose acoustic embeddings are nearest to a user’s listening history, all within a single query to the vector database. These patterns demonstrate that vector stores are not merely a drop‑in replacement for inverted indexes; they are a unifying data layer that powers richer, faster, and more intuitive search experiences.
